<a href="https://colab.research.google.com/github/mikecrv2019-bit/MAESTRIA-IA/blob/main/Tarea_06_Telco_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telco Customer Churn — Tarea 06

**MAI540 — Machine Learning · Atlantis University**

Comparación entre el modelo de partida (3 variables) y la versión equilibrada
solicitada en el README (10 variables, `ColumnTransformer`, `class_weight="balanced"`).

Ambos modelos usan `test_size=0.25`, `random_state=42` y `stratify=y`, de modo que
se evalúan sobre el mismo conjunto de prueba.

## 1. Cargar el dataset

Ejecuta esta celda y sube el archivo `datos.csv` cuando aparezca el botón.
Si el archivo ya está en la sesión, la celda no pedirá nada.

In [ ]:
from pathlib import Path

DATA = "datos.csv"

if not Path(DATA).exists():
    from google.colab import files
    print("Selecciona el archivo datos.csv de tu computadora:")
    files.upload()

print("Archivo listo:", Path(DATA).exists())

Archivo listo: True


## 2. Importaciones

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv(DATA)
print(f"Filas cargadas: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

Filas cargadas: 4,500
Columnas: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## 3. Modelo ANTES — punto de partida

Código original sin modificar: 3 variables, imputación por mediana, sin escalado
y sin balanceo de clases. Sirve como línea de base para la comparación.

In [ ]:
features = ['SeniorCitizen', 'tenure', 'MonthlyCharges']
X = df[features]
y = (df["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("=== TELCO CUSTOMER CHURN: PUNTO DE PARTIDA (ANTES) ===")
print(f"Filas del dataset: {len(df):,}")
print(f"Tamano de X_test: {len(X_test):,}")
print(f"Variables usadas por el modelo inicial: {len(features)}")

antes_accuracy = accuracy_score(y_test, pred)
antes_precision = precision_score(y_test, pred, pos_label=1)
antes_recall = recall_score(y_test, pred, pos_label=1)
antes_f1 = f1_score(y_test, pred, pos_label=1)

print(f"Accuracy: {antes_accuracy:.4f}")
print(f"Precision (Churn=Yes): {antes_precision:.4f}")
print(f"Recall (Churn=Yes): {antes_recall:.4f}")
print(f"F1 (Churn=Yes): {antes_f1:.4f}")
print("Matriz de confusion:")
print(confusion_matrix(y_test, pred))

=== TELCO CUSTOMER CHURN: PUNTO DE PARTIDA (ANTES) ===
Filas del dataset: 4,500
Tamano de X_test: 1,125
Variables usadas por el modelo inicial: 3
Accuracy: 0.8000
Precision (Churn=Yes): 0.6850
Recall (Churn=Yes): 0.4582
F1 (Churn=Yes): 0.5491
Matriz de confusion:
[[763  63]
 [162 137]]


## 4. Modelo DESPUÉS — versión equilibrada

Diez variables. `TotalCharges` se convierte de texto a número; los valores que no
se pueden convertir quedan como `NaN` y se imputan dentro del pipeline, sin eliminar filas.

El `train_test_split` ocurre **antes** de cualquier ajuste, de modo que el escalado,
la imputación y el one-hot encoding se calculan únicamente con los datos de
entrenamiento. El conjunto de prueba solo se transforma.

In [ ]:
df_full = df.copy()
df_full["TotalCharges"] = pd.to_numeric(df_full["TotalCharges"], errors="coerce")

numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [
    "SeniorCitizen", "Partner", "Dependents",
    "InternetService", "OnlineSecurity", "Contract", "PaymentMethod",
]
full_features = numeric_features + categorical_features

X_full = df_full[full_features]
y_full = (df_full["Churn"] == "Yes").astype(int)

X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full, y_full, test_size=0.25, random_state=42, stratify=y_full
)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

full_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])
full_model.fit(X_full_train, y_full_train)
full_pred = full_model.predict(X_full_test)

print("=== TELCO CUSTOMER CHURN: VERSION EQUILIBRADA (DESPUES) ===")
print(f"Filas del dataset: {len(df_full):,}")
print(f"Tamano de X_test: {len(X_full_test):,}")
print(f"Variables usadas por el modelo final: {len(full_features)}")

despues_accuracy = accuracy_score(y_full_test, full_pred)
despues_precision = precision_score(y_full_test, full_pred, pos_label=1)
despues_recall = recall_score(y_full_test, full_pred, pos_label=1)
despues_f1 = f1_score(y_full_test, full_pred, pos_label=1)

print(f"Accuracy: {despues_accuracy:.4f}")
print(f"Precision (Churn=Yes): {despues_precision:.4f}")
print(f"Recall (Churn=Yes): {despues_recall:.4f}")
print(f"F1 (Churn=Yes): {despues_f1:.4f}")
print("Matriz de confusion:")
print(confusion_matrix(y_full_test, full_pred))

=== TELCO CUSTOMER CHURN: VERSION EQUILIBRADA (DESPUES) ===
Filas del dataset: 4,500
Tamano de X_test: 1,125
Variables usadas por el modelo final: 10
Accuracy: 0.7333
Precision (Churn=Yes): 0.4989
Recall (Churn=Yes): 0.7559
F1 (Churn=Yes): 0.6011
Matriz de confusion:
[[599 227]
 [ 73 226]]


## 5. Comparación ANTES vs DESPUÉS

In [ ]:
print("=== COMPARACION ANTES vs DESPUES ===")
print(f"{'Metrica':<25}{'ANTES':>10}{'DESPUES':>10}")
print(f"{'Filas del dataset':<25}{len(df):>10,}{len(df_full):>10,}")
print(f"{'Tamano de X_test':<25}{len(X_test):>10,}{len(X_full_test):>10,}")
print(f"{'Variables usadas':<25}{len(features):>10}{len(full_features):>10}")
print(f"{'Accuracy':<25}{antes_accuracy:>10.4f}{despues_accuracy:>10.4f}")
print(f"{'Precision (Yes)':<25}{antes_precision:>10.4f}{despues_precision:>10.4f}")
print(f"{'Recall (Yes)':<25}{antes_recall:>10.4f}{despues_recall:>10.4f}")
print(f"{'F1 (Yes)':<25}{antes_f1:>10.4f}{despues_f1:>10.4f}")

=== COMPARACION ANTES vs DESPUES ===
Metrica                       ANTES   DESPUES
Filas del dataset             4,500     4,500
Tamano de X_test              1,125     1,125
Variables usadas                  3        10
Accuracy                     0.8000    0.7333
Precision (Yes)              0.6850    0.4989
Recall (Yes)                 0.4582    0.7559
F1 (Yes)                     0.5491    0.6011


## 6. Qué cambió y por qué

## Los cambios que se presentaron fueron los siguientes:

##•	De utilizar 3 variable a usar (10) variables, se agregó la (Columna Transforme) que trata de forma numérica y categorías por separado agregando (class weight=balanced)

##•	El recall subio de 0.4582 a 0.7559, de 137 clientes detectados a 226 de un total de 299 que había cancelados, el F1 de 0,5491 subio a 0.6011. Bajando la precisión de 0.6850 a 04989, aunque el accuracy bajo de 0.800 se redujo a 0.7333

##•	El modelo no tenía incentivo para detectar a los que sí cancelan por la mayoría de los clientes que cancelan. Class weight = Balanced hace que los errores sobre la clase minoritaria pesen más en el entrenamiento, así que el modelo se vuelve más agresivo detectándolos a costa de marcar como riesgo a clientes que no lo son.

##•	Aun que mejora el accuracy, demuestra criterios de retención un falso positivo cuesta una llamada comercial un falso negativo cuesta la pérdida de un cliente. F1 combina precision y recall, subió asi que no es solo un intercambio, el desempeño conjunto sobre la clase que importa mejoro considerablemente.


